## Summary

In this notebook, we:
- Integrated an LLM to simplify legal clauses
- Designed prompt templates for consistent outputs
- Combined clause type, risk level, and simplified explanation
- Generated human-readable legal insights

Next notebook:
➡ 06_final_pipeline.ipynb


In [ ]:
import pandas as pd
import re
import pickle
import os
import sys

sys.path.append("..")

from src.simplification import simplify_clause


In [10]:
# Load clause classification model
with open("../models/clause_classifier/tfidf.pkl", "rb") as f:
    tfidf = pickle.load(f)

with open("../models/clause_classifier/model.pkl", "rb") as f:
    classifier = pickle.load(f)

print("Models loaded successfully.")

Models loaded successfully.


In [11]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()


In [12]:
def segment_clauses(text):
    clauses = re.split(r"\.\s+|;\s+|\n+|\:\s+", text)
    clauses = [c.strip() for c in clauses if len(c.strip()) > 20]
    return clauses


In [13]:
RISK_KEYWORDS = {
    "high": ["penalty", "terminate immediately", "unlimited liability", "sole discretion"],
    "medium": ["terminate", "liability", "damages", "breach"],
    "low": ["notice", "agreement", "confidential"]
}

def calculate_risk_score(text):
    score = 0
    text = text.lower()
    for word in RISK_KEYWORDS["high"]:
        if word in text:
            score += 3
    for word in RISK_KEYWORDS["medium"]:
        if word in text:
            score += 2
    for word in RISK_KEYWORDS["low"]:
        if word in text:
            score += 1
    return score

def risk_level(score):
    if score >= 6:
        return "High Risk"
    elif score >= 3:
        return "Medium Risk"
    else:
        return "Low Risk"


In [15]:
!pip install -q groq
import os
from groq import Groq

# Set Groq API key
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def build_prompt(clause):
    return f"""
Simplify the following legal clause into plain English.
Avoid legal jargon. Be concise.

Clause:
\"\"\"{clause}\"\"\"

Simplified Explanation:
"""



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\ADITYA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [21]:
def simplify_clause(clause_text):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # Valid Groq model
        messages=[
            {"role": "user", "content": build_prompt(clause_text)}
        ],
        temperature=0.3,
        max_tokens=150
    )
    
    return response.choices[0].message.content.strip()

In [ ]:
def analyze_document(raw_text, max_clauses=8):
    cleaned = clean_text(raw_text)
    clauses = segment_clauses(cleaned)

    results = []

    for clause in clauses[:max_clauses]:
        vec = tfidf.transform([clause])
        clause_type = classifier.predict(vec)[0]

        score = calculate_risk_score(clause)
        level = risk_level(score)

        simplified = simplify_clause(clause)

        results.append({
            "clause_text": clause,
            "clause_type": clause_type,
            "risk_score": score,
            "risk_level": level,
            "simplified_text": simplified
        })

    return pd.DataFrame(results)


In [23]:
sample_text = """
Either party may terminate this agreement with immediate effect
in case of material breach. The company shall not be liable for
any indirect or consequential damages. All confidential information
must be kept secret for a period of five years.
"""

final_output = analyze_document(sample_text)

final_output


,clause_text,clause_type,risk_score,risk_level,simplified_text
0,either party may terminate this agreement with...,Cap On Liability,8,High Risk,Either party can end this agreement immediatel...


In [24]:
os.makedirs("../outputs", exist_ok=True)

final_output.to_csv(
    "../outputs/final_pipeline_output.csv",
    index=False
)

print("Final pipeline output saved.")


Final pipeline output saved.


## Final Summary

This notebook demonstrates:
- End-to-end legal text processing
- Clause segmentation and classification
- Risk scoring using explainable rules
- LLM-based legal simplification
- A unified pipeline ready for deployment

This pipeline can be directly integrated into a Streamlit app.
